# 🌌 LingBot-World: Discover & Simulate New Worlds
Welcome to the **LingBot-World Colab Notebook**! This notebook is configured to run the state-of-the-art **LingBot-World** world simulator on Google Colab's cloud GPUs (T4, L4, or A100).

### 🎮 Connect to your Playgound
You can design custom camera paths in the local playground I built for you, copy the **Compact Action String**, and paste it into the form below to drive your world generation!

## 🛠️ Step 1: Install Dependencies
Run the cell below to clone the repository, install Python packages, and set up the environment.

In [ ]:
#@title Setup Environment
!git clone https://github.com/robbyant/lingbot-world.git
%cd lingbot-world

print("Installing dependencies (this may take a couple of minutes)...")
!pip install -r requirements.txt
!pip install "huggingface_hub[cli]"

# Pre-create directories
!mkdir -p examples/custom_run

## 📥 Step 2: Download Model Weights
Select which version of the model to download. For free Google Colab instances (T4 GPU), we recommend using the **4-bit quantized version** to prevent Out Of Memory (OOM) errors.

In [ ]:
#@title Select & Download Model Weights
model_version = "4-bit Quantized (Recommended for free T4)" #@param ["4-bit Quantized (Recommended for free T4)", "Full Precision Base (Requires A100/L4)"]

if "4-bit" in model_version:
    print("Downloading 4-bit Quantized Model...")
    !huggingface-cli download cahlen/lingbot-world-base-cam-nf4 --local-dir ./lingbot-world-base-cam
else:
    print("Downloading Full Precision Base Model...")
    !huggingface-cli download robbyant/lingbot-world-base-cam --local-dir ./lingbot-world-base-cam

print("Model weights downloaded successfully!")

## 🖼️ Step 3: Upload Starter Image
Upload the first frame of your video (the starting image). If you don't upload one, a default fantasy jungle image will be used.

In [ ]:
from google.colab import files
import os

print("Please select your starter image (.jpg or .png):")
uploaded = files.upload()

if uploaded:
    image_name = list(uploaded.keys())[0]
    target_path = "examples/custom_run/image.jpg"
    os.rename(image_name, target_path)
    print(f"Uploaded {image_name} successfully as starting frame.")
else:
    print("No image uploaded. Using default example image.")
    !cp examples/05/image.jpg examples/custom_run/image.jpg

## 🎬 Step 4: Configure & Generate Video
Configure your prompt and paste the action string you designed from the playground!

In [ ]:
#@title Generation Parameters
prompt = "A serene soaring flight through a fantasy jungle, mist rising from the trees, sunrays breaking through" #@param {type:"string"}
action_string = "w-15,j-10,s-15,l-10" #@param {type:"string"}
resolution = "480*832" #@param ["480*832", "720*1280"]
sampling_steps = 20 #@param {type:"slider", min:10, max:50, step:1}

# Create default camera intrinsics for our custom trajectory
import numpy as np
from wan.utils.wasd_ijkl_to_c2ws import action_string_to_wasd_ijkl, pad_frame_num_to_4n_plus_1

_, _, str_frames = action_string_to_wasd_ijkl(action_string)
padded_frames = pad_frame_num_to_4n_plus_1(str_frames)

# Create default intrinsics matrix
intrinsics = np.zeros((padded_frames, 4), dtype=np.float32)
intrinsics[:, 0] = 480.0  # fx
intrinsics[:, 1] = 480.0  # fy
intrinsics[:, 2] = 416.0  # cx
intrinsics[:, 3] = 240.0  # cy
np.save("examples/custom_run/intrinsics.npy", intrinsics)

print(f"Configuration ready. Total frames implied: {padded_frames}.")

### 🚀 Run Generation

In [ ]:
# Run generation script
!python generate.py \
  --task i2v-A14B \
  --size {resolution} \
  --ckpt_dir lingbot-world-base-cam \
  --image examples/custom_run/image.jpg \
  --action_path examples/custom_run \
  --action_string "{action_string}" \
  --allow_act2cam \
  --sample_steps {sampling_steps} \
  --offload_model True \
  --t5_cpu \
  --save_file "output_world.mp4" \
  --prompt "{prompt}"

### 📺 Display Result

In [ ]:
from ipywidgets import Video
import os

if os.path.exists("output_world.mp4"):
    display(Video.from_file("output_world.mp4", width=640, height=360))
else:
    print("Output video file not found. Check generator logs above.")